#  Machine Learning Pipeline

Machine learning models require the data to undergo several preprocessing steps before training. Throughout the preprocessing phase of this project, we determined how the dataset should be prepared by performing feature engineering, handling missing values, encoding categorical variables, and applying feature scaling.

The next step is to organize these preprocessing operations into a structured and reusable workflow known as a **Machine Learning Pipeline**.

A pipeline automates the sequence of preprocessing steps and model training so that every dataset is processed consistently before being passed to the learning algorithm. Instead of manually executing each preprocessing step every time a new model is trained or new data is received, the entire workflow can be executed automatically through a single pipeline.

This improves consistency, reduces repetitive code, minimizes implementation errors, and makes the machine learning workflow easier to maintain and reproduce.

---

## Step 1: What is a Machine Learning Pipeline?

A Machine Learning Pipeline is a structured workflow that combines multiple preprocessing operations and a machine learning model into a single automated process.

Rather than treating feature engineering, missing value handling, encoding, feature scaling, and model training as independent tasks, a pipeline connects them together in a predefined sequence. Each step receives the output of the previous step, performs a specific transformation, and passes the processed data to the next stage until the final model is trained or predictions are generated.

The pipeline therefore acts as a workflow manager that ensures every dataset follows the same preprocessing sequence before reaching the machine learning algorithm.

---

## Step 2: Why do we need a Machine Learning Pipeline?

During this project, we identified several preprocessing operations that must be performed before training a machine learning model, including feature engineering, missing value handling, encoding categorical variables, and feature scaling.

Without a pipeline, these preprocessing steps would need to be performed manually each time a model is trained. This leads to repetitive code, increases the possibility of human error, and makes the workflow difficult to maintain.

A Machine Learning Pipeline solves this problem by automating the entire preprocessing workflow. Once the pipeline is defined, the required preprocessing steps are executed automatically in the correct order whenever a model is trained or used for prediction.

Different machine learning algorithms may require different preprocessing pipelines. For example, linear models typically require feature scaling, whereas tree-based models generally do not. Therefore, pipelines are designed according to the preprocessing requirements of a particular family of models.

Within a given model family, all models share the same preprocessing pipeline and are trained on identically prepared data. As a result, differences in model performance can be attributed to the learning algorithm itself rather than inconsistencies in data preparation, making model comparisons fair, reliable, and reproducible.


---

## Step 3: Different Types of Machine Learning Pipelines

There is no universal Machine Learning Pipeline that is suitable for every algorithm. The design of a pipeline depends on the preprocessing requirements of the machine learning model being trained.

Different algorithms learn from data in different ways. Consequently, they may require different preprocessing steps before training.

For example, linear models such as Linear Regression, Ridge Regression, and Lasso Regression optimize model parameters using gradient-based optimization. Since these algorithms are sensitive to differences in feature magnitudes, feature scaling is an essential part of their preprocessing pipeline.

In contrast, tree-based algorithms such as Decision Trees, Random Forests, and XGBoost learn by recursively splitting the feature space rather than optimizing numerical weights. Because these models are not influenced by the scale of input features, feature scaling is generally unnecessary and is therefore excluded from their preprocessing pipeline.

Although different model families may use different preprocessing pipelines, this does not make model comparison unfair. A fair comparison does not require every algorithm to use an identical pipeline. Instead, each model should be evaluated using the preprocessing steps that are appropriate for its learning mechanism.

To ensure a reliable comparison, all other experimental conditions should remain consistent, including the train-validation split, feature engineering, missing value handling, encoding strategy, and evaluation metrics. Only the preprocessing steps that are inherently required by a particular algorithm should differ.

Designing pipelines according to the requirements of each model family ensures that every algorithm is evaluated under its appropriate conditions while maintaining consistency, reproducibility, and fairness throughout the machine learning workflow.

---

## Step 4: Designing the Pipeline for Our Project

After completing the preprocessing phase, we have already determined how the dataset should be prepared for machine learning. Therefore, the objective of this step is not to decide new preprocessing techniques, but to organize the previously selected transformations into a structured, reusable, and production-ready workflow.

Before training multiple machine learning algorithms, it is a common practice to first establish a **baseline model**. A baseline model is the first and simplest model used to provide a reference point for evaluating more advanced models. Rather than immediately using complex algorithms, a baseline model allows us to understand how well a simple approach performs and serves as a benchmark against which future models can be compared.

For this project, **Linear Regression** is selected as the baseline model because it is computationally efficient, easy to interpret, and widely accepted as the standard starting point for regression problems. After establishing its performance, more sophisticated models such as Ridge Regression, Lasso Regression, Random Forest, and XGBoost can be evaluated to determine whether the additional complexity results in meaningful improvements.

Since Linear Regression is sensitive to differences in feature magnitudes, the preprocessing pipeline is designed according to the requirements of linear models.

The workflow of our pipeline is shown below.

```
Raw Data
      ↓
Train / Validation Split
      ↓
Missing Value Handling
      ↓
Encoding
      ↓
Feature Engineering
      ↓
Feature Scaling (StandardScaler)
      ↓
Linear Regression (Baseline Model)
```

### Why this pipeline order?

The order of preprocessing steps is not determined by a fixed universal rule. Instead, it is decided based on the **dependencies between transformations**. Every preprocessing step should be placed only after all the information it requires is available.

### 1. Train / Validation Split

The dataset is first divided into training and validation sets to prevent **data leakage**. Any preprocessing operation that learns information from the data—such as missing value imputation, encoding, or feature scaling—must be fitted only on the training data. The learned parameters are then reused to transform the validation and future unseen data.

### 2. Missing Value Handling

Missing values are handled before any further transformations to ensure that subsequent preprocessing steps operate on complete and reliable data. Performing feature engineering or other transformations on incomplete data may propagate missing values into newly created features, resulting in unnecessary information loss.

### 3. Encoding

Categorical variables are converted into numerical representations before feature engineering. Although some engineered features can be created directly from numerical columns, placing encoding before feature engineering provides a consistent numerical representation whenever engineered features depend on categorical variables. This also makes the pipeline easier to extend and maintain as additional engineered features are introduced in the future.

### 4. Feature Engineering

Feature engineering is performed after the dataset has been cleaned and categorical variables have been converted into numerical representations. At this stage, all required information is available, allowing meaningful features to be constructed from both numerical and encoded categorical variables. In this project, engineered features such as **HouseAge**, **TotalSF**, and **TotalBath** are created to provide more informative representations of the original data.

### 5. Feature Scaling

Feature scaling is intentionally performed after feature engineering because scaling should be applied to the **final set of numerical features**. If scaling were performed before feature engineering, any newly created numerical features would remain unscaled, leading to inconsistent feature magnitudes. Since Linear Regression is sensitive to differences in feature scales, **StandardScaler** is applied as the final preprocessing step before model training.

By organizing the preprocessing steps according to their dependencies, the pipeline ensures that every transformation receives appropriate input data while maintaining consistency, preventing data leakage, and providing a reusable workflow for training and inference.

---

## Step 5: Implementing the Pipeline using Scikit-learn

Scikit-learn provides the `Pipeline` class to combine multiple preprocessing operations and a machine learning model into a single executable workflow.

Instead of manually performing each preprocessing step before training the model, a pipeline automatically executes every transformation in the predefined order. This simplifies the training process, reduces repetitive code, and ensures that the same preprocessing workflow is consistently applied during both training and prediction.

A Scikit-learn pipeline consists of a sequence of **intermediate transformers** followed by a **final estimator**.

- **Intermediate transformers** perform preprocessing operations such as missing value handling, encoding, feature engineering, and feature scaling.
- The **final estimator** is the machine learning model that learns the relationship between the processed features and the target variable.

### How does a Pipeline work?

Every intermediate transformer in Scikit-learn follows two fundamental operations:

- **`fit()`** – Learns the required parameters from the training data.
- **`transform()`** – Applies the learned parameters to transform the data.

This separation exists because most preprocessing algorithms first need to **learn** information from the training dataset before they can consistently **apply** the same transformation to any dataset.

For example:

- A **SimpleImputer** learns the median or most frequent value.
- A **StandardScaler** learns the mean and standard deviation.
- An **Encoder** learns the categories or mappings present in the training data.

Once these parameters have been learned during `fit()`, they are stored inside the transformer and reused whenever `transform()` is called.

This design allows the same fitted transformer to be applied to both the training and validation (or test) datasets, ensuring that every sample is represented in exactly the same feature space.

Separating the learning (`fit`) and transformation (`transform`) phases also helps prevent **data leakage**. Since all preprocessing parameters are learned exclusively from the training data, the validation or test data never influences the preprocessing process. Instead, the already-fitted transformer is reused to transform unseen data. This ensures that the model is evaluated on data represented in the same way as the data it was trained on, resulting in a fair and reliable evaluation.

The final estimator differs slightly from intermediate transformers. Instead of implementing `transform()`, it learns the relationship between the processed features and the target variable using `fit()`, and generates predictions using `predict()`.

The complete workflow of a Scikit-learn Pipeline is therefore:

```
Intermediate Transformer
        │
     fit()
        │
 transform()
        ▼
Intermediate Transformer
        │
     fit()
        │
 transform()
        ▼
Intermediate Transformer
        │
     fit()
        │
 transform()
        ▼
Final Estimator
        │
      fit()
        │
    predict()
```

By combining preprocessing and model training into a single object, a pipeline provides a clean, reusable, production-ready workflow that ensures preprocessing is applied consistently, minimizes human error, prevents data leakage, and simplifies model training and prediction.

---

## Step 6: Import Required Libraries

After designing the preprocessing pipeline, the next step is to import the Scikit-learn components required for its implementation.

Our pipeline consists of several preprocessing operations followed by a machine learning model. Before constructing the pipeline, the dataset must first be divided into training and validation sets to ensure that all preprocessing transformations learn only from the training data. Each preprocessing operation is implemented using a dedicated Scikit-learn transformer, while the entire workflow is organized using the `Pipeline` and `ColumnTransformer` classes.

The primary components required for this implementation are:

- **train_test_split** – Splits the dataset into training and validation sets before any preprocessing is performed, preventing data leakage by ensuring that all preprocessing transformations are learned only from the training data.
- **Pipeline** – Chains multiple preprocessing steps and the final estimator into a single executable workflow.
- **ColumnTransformer** – Applies different preprocessing pipelines to numerical and categorical features.
- **SimpleImputer** – Handles missing values.
- **OneHotEncoder** – Converts categorical variables into numerical representations.
- **StandardScaler** – Standardizes numerical features for linear models.
- **LinearRegression** – Serves as the baseline regression model.

These components together enable the construction of a clean, reusable, and production-ready preprocessing workflow.

In [1]:
# ============================================================
# Data Splitting
# ============================================================

from sklearn.model_selection import train_test_split

# ============================================================
# Pipeline Construction
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ============================================================
# Preprocessing
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# ============================================================
# Machine Learning Model
# ============================================================

from sklearn.linear_model import LinearRegression

## Step 7: Understanding ColumnTransformer

Our dataset contains two different types of features: **numerical** and **categorical**. Since these feature types have different characteristics, they require different preprocessing techniques before they can be used to train a machine learning model.

For example:

- **Numerical features** require operations such as missing value imputation and feature scaling.
- **Categorical features** require missing value imputation followed by categorical encoding.

Because of these different preprocessing requirements, applying a single preprocessing pipeline to the entire dataset is not appropriate. A scaler should not be applied to categorical features, and an encoder should not be applied to numerical features.

To solve this problem, Scikit-learn provides the **`ColumnTransformer`** class.

`ColumnTransformer` applies different preprocessing pipelines to different subsets of columns within the same dataset. Each group of features is processed independently using its corresponding preprocessing pipeline, after which the transformed outputs are automatically combined into a single processed feature matrix.

The overall workflow can be visualized as follows:

```
                    Dataset
                       │
            ┌──────────┴──────────┐
            │                     │
            ▼                     ▼
    Numerical Features     Categorical Features
            │                     │
            ▼                     ▼
   Numerical Pipeline     Categorical Pipeline
            │                     │
            └──────────┬──────────┘
                       ▼
               ColumnTransformer
                       │
                       ▼
         Combined Processed Feature Matrix
```

Using `ColumnTransformer` provides a clean and modular approach to preprocessing heterogeneous datasets. It ensures that every feature is transformed using the preprocessing technique appropriate for its data type while producing a single processed dataset that can be directly used for model training.

---

## Step 8: Loading the Dataset

In [2]:
import pandas as pd
#Load DataSet
df=pd.read_csv('../data/train.csv')

In [3]:
df.shape

(1460, 81)

## Step 9: Train-Validation Split

After loading the dataset, the next step is to divide it into separate **training** and **validation** sets.

This split is performed **before** constructing the preprocessing pipeline because several preprocessing operations, such as missing value imputation, categorical encoding, and feature scaling, learn information from the data during the `fit()` stage.

If these transformations were fitted on the entire dataset before splitting, information from the validation set would unintentionally influence the learned preprocessing parameters. This would introduce **data leakage**, resulting in an overly optimistic estimate of the model's performance.

To prevent this, the dataset is first divided into training and validation sets. All preprocessing components within the pipeline will later be fitted exclusively on the training data, while the same fitted transformations will be applied to the validation data. This ensures that the validation set remains completely unseen during training, providing a fair and reliable evaluation of the model's ability to generalize to unseen data.

For this project:

- **80%** of the data is used for training.
- **20%** of the data is reserved for validation.
- A fixed `random_state` is specified to ensure reproducibility, meaning that the same train-validation split is generated every time the notebook is executed.
---

In [4]:
# Separate Features and Target Variable
X = df.drop(columns=["SalePrice","Id"])
y = df["SalePrice"]

# Train-Validation Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [5]:
print(f"Training Features   : {X_train.shape}")
print(f"Validation Features : {X_valid.shape}")

print()

print(f"Training Target     : {y_train.shape}")
print(f"Validation Target   : {y_valid.shape}")

Training Features   : (1168, 79)
Validation Features : (292, 79)

Training Target     : (1168,)
Validation Target   : (292,)


## Step 10: Identifying Numerical and Categorical Features

After splitting the dataset into training and validation sets, the next step is to identify the numerical and categorical features.

This separation is necessary because different feature types require different preprocessing techniques. During the preprocessing phase of this project, we determined that numerical features require missing value imputation followed by feature scaling, whereas categorical features require missing value imputation followed by categorical encoding.

Scikit-learn's `ColumnTransformer` does not automatically determine which preprocessing pipeline should be applied to each feature. Instead, it requires the numerical and categorical feature lists to be explicitly specified. These feature lists are then used to route each group of features to its corresponding preprocessing pipeline.

The feature groups are identified using the training dataset (`X_train`). Although the data types remain the same in both the training and validation datasets, consistently using the training data ensures that every preprocessing component is designed based only on the data available during model training.

Identifying the feature groups before constructing the preprocessing pipelines also improves code readability, maintainability, and makes the workflow easier to extend if additional features are introduced in the future.

---

In [6]:
# Identify Numerical and Categorical Features

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["str"]
).columns.tolist()

In [7]:
print("Numerical Features")
print(numerical_features)

print()

print("Categorical Features")
print(categorical_features)

Numerical Features
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

Categorical Features
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType',

---

## Step 11: Building the Numerical Missing Value Transformer

The first component of the numerical preprocessing pipeline is the **Numerical Missing Value Transformer**.

During the preprocessing phase of this project, several numerical features were found to contain missing values. However, these missing values could not be handled using a single generic imputation strategy because each feature required a different preprocessing decision based on exploratory data analysis and domain knowledge.

The numerical missing value handling decisions were:

| Feature | Imputation Strategy |
|---------|---------------------|
| `LotFrontage` | Neighborhood-wise median |
| `MasVnrArea` | `0` |
| `GarageYrBlt` | `0` |

While Scikit-learn provides generic transformers such as `SimpleImputer`, they cannot directly implement this combination of custom preprocessing rules. Therefore, a custom transformer is created by inheriting from Scikit-learn's `BaseEstimator` and `TransformerMixin`.

This custom transformer follows the standard Scikit-learn estimator interface by implementing two methods:

- **`fit()`** – Learns the statistics required from the training data.
- **`transform()`** – Applies the learned statistics and fixed preprocessing rules to both training and unseen data.

Separating the learning and transformation stages ensures that all imputation statistics are learned **only from the training data**, preventing data leakage and guaranteeing consistent preprocessing across the training, validation, and future production datasets.

The implementation of this transformer is completed in the following two steps:

- **Step 11.1:** Learning the Missing Value Statistics (`fit()`)
- **Step 11.2:** Applying the Missing Value Transformations (`transform()`)

---

### Step 11.1: Learning the Missing Value Statistics

The `fit()` method is responsible for learning any statistics required by the transformer from the **training data only**.

For this project, only one numerical feature requires learning from the data:

- **LotFrontage** → Neighborhood-wise median

During exploratory data analysis, it was observed that the frontage of a property varies significantly across different neighborhoods. Therefore, instead of replacing missing values using a single global median, the median frontage is computed separately for each neighborhood.

These neighborhood-specific median values are learned during the `fit()` stage and stored within the transformer. They are later reused during the `transform()` stage for both the training and validation datasets.

Learning these statistics only from the training data prevents data leakage and ensures that the validation and future test datasets do not influence the preprocessing decisions.

---

### Step 11.2: Applying the Missing Value Transformations

After learning the required imputation statistics during the `fit()` stage, the `transform()` method applies the missing value handling rules to the dataset.

The transformer follows the numerical missing value handling decisions established during the preprocessing phase:

- `MasVnrArea` is replaced with **0**, indicating the absence of masonry veneer.
- `GarageYrBlt` is replaced with **0**, indicating the absence of a garage.
- Missing values in `LotFrontage` are replaced using the **Neighborhood-wise median** learned during the `fit()` stage.

The transformer first creates a copy of the input dataset before applying any modifications. This ensures that the original dataset remains unchanged while the transformed copy is returned for the subsequent preprocessing steps.

Separating the learning (`fit`) and application (`transform`) stages ensures that the same preprocessing logic is consistently applied to the training, validation, and future unseen datasets without introducing data leakage.

---

### Why `BaseEstimator` and `TransformerMixin`?

To make our custom preprocessing class behave like a built-in Scikit-learn transformer, we inherit from `BaseEstimator` and `TransformerMixin`.

- **`BaseEstimator`** makes the class compatible with the Scikit-learn API by providing features such as parameter management (`get_params()` and `set_params()`). This allows the transformer to work seamlessly with utilities like `Pipeline` and `GridSearchCV`.
- **`TransformerMixin`** automatically provides the `fit_transform()` method by combining the implemented `fit()` and `transform()` methods, so we don't need to implement it ourselves.

By inheriting from these two classes, our custom transformer can be used just like any built-in Scikit-learn transformer inside a machine learning pipeline.

---

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin


class NumericalMissingValueTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):

        # Learn Neighborhood-wise median for LotFrontage
        self.lotfrontage_median_ = (
            X.groupby("Neighborhood")["LotFrontage"]
             .median()
        )

        return self

    def transform(self, X):

        X = X.copy()


        # MasVnrArea
      

        X["MasVnrArea"] = X["MasVnrArea"].fillna(0)

     
        # GarageYrBlt
        

        X["GarageYrBlt"] = X["GarageYrBlt"].fillna(0)

        # LotFrontage

        X["LotFrontage"] = X["LotFrontage"].fillna(
            X["Neighborhood"].map(self.lotfrontage_median_)
        )

        return X

In [9]:
# Create transformer
numerical_missing_transformer = NumericalMissingValueTransformer()

# Learn statistics from training data
numerical_missing_transformer.fit(X_train)

# Apply transformations
X_train_num = numerical_missing_transformer.transform(X_train)

# Verify missing values
X_train_num[
    ["LotFrontage", "MasVnrArea", "GarageYrBlt"]
].isnull().sum()

LotFrontage    0
MasVnrArea     0
GarageYrBlt    0
dtype: int64

----

## Step 12: Building the Categorical Missing Value Transformer

The next component of the preprocessing pipeline is the **Categorical Missing Value Transformer**.

During exploratory data analysis, missing values in categorical features were found to represent two different situations:

1. **Feature Absence** – The missing value indicates that the corresponding property does not exist (for example, no garage, no basement, or no pool).
2. **Incomplete Data** – The feature exists, but one or more related attributes are missing due to data inconsistency or incomplete recording.

Based on these observations, the preprocessing strategy combines **rule-based data cleaning** with **statistical imputation**.

- Features representing **feature absence** are replaced with `"None"`.
- Certain basement-related inconsistencies are handled using predefined business rules before feature-absence imputation.
- Features requiring statistical imputation are filled using values learned only from the training data.

Since these preprocessing decisions cannot be implemented using a single generic imputer, a custom Scikit-learn transformer is created. The transformer encapsulates the entire categorical missing value handling logic while remaining fully compatible with the Scikit-learn pipeline.

Like every Scikit-learn transformer, it implements the `fit()` and `transform()` methods. Any statistical parameters are learned only from the training data during the `fit()` stage and are consistently reused during the `transform()` stage, ensuring reproducible preprocessing and preventing data leakage.

---

### Step 12.1: Learning the Missing Value Statistics

The `fit()` method learns the statistical values required for categorical imputation from the **training data only**.

In this project, only the following features require statistical imputation:

- `BsmtFinType2` → Mode
- `Electrical` → Mode

These statistics are computed during the `fit()` stage and stored within the transformer. During the `transform()` stage, the same learned values are reused for the training, validation, test, and future unseen datasets.

Learning preprocessing parameters exclusively from the training data ensures consistent preprocessing while preventing data leakage.

---

### Step 12.2: Applying the Missing Value Transformations

After learning the required statistical values during the `fit()` stage, the `transform()` method applies the categorical missing value handling rules.

The transformation is performed in the following order:

1. **Rule-based Data Cleaning**

   Certain basement-related missing values indicate incomplete records rather than the absence of a basement. These cases are corrected using predefined business rules derived during exploratory data analysis.

   - If a basement exists but `BsmtExposure` is missing, it is replaced with `"No"`.
   - If a basement exists but `BsmtFinType2` is missing, it is imputed using the mode learned during the `fit()` stage.

2. **Feature Absence Handling**

   The remaining missing values correspond to the absence of a property (such as no garage, basement, pool, or masonry veneer) and are therefore replaced with `"None"`.

3. **Statistical Imputation**

   The missing value in `Electrical` is imputed using the mode learned from the training data.

Before applying any transformations, a copy of the input dataset is created to ensure that the original data remains unchanged.

By separating **rule-based corrections**, **feature-absence handling**, and **statistical imputation**, the transformer provides a clean, reusable, and production-ready preprocessing workflow that can be consistently applied to training, validation, test, and future unseen datasets without introducing data leakage.

---

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin


class CategoricalMissingValueTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):

        # Learn mode values from training data
        self.bsmtfintype2_mode_ = X["BsmtFinType2"].mode()[0]
        self.electrical_mode_ = X["Electrical"].mode()[0]

        return self

    def transform(self, X):

        X = X.copy()

        # Correct ambiguous basement records identified during EDA

        X.loc[
            X["BsmtExposure"].isna() & X["BsmtQual"].notna(),
            "BsmtExposure"
        ] = "No"

        X.loc[
            X["BsmtFinType2"].isna() & X["BsmtQual"].notna(),
            "BsmtFinType2"
        ] = self.bsmtfintype2_mode_

        # Feature Absence -> "None"

        feature_absence = [
            "PoolQC",
            "MiscFeature",
            "Alley",
            "Fence",
            "FireplaceQu",
            "GarageType",
            "GarageFinish",
            "GarageQual",
            "GarageCond",
            "BsmtQual",
            "BsmtCond",
            "BsmtExposure",
            "BsmtFinType1",
            "BsmtFinType2",
            "MasVnrType",
        ]

        X[feature_absence] = X[feature_absence].fillna("None")

        # ==========================================================
        # Electrical
        # ==========================================================

        X["Electrical"] = X["Electrical"].fillna(
            self.electrical_mode_
        )

        return X

In [11]:
# Create transformer
categorical_missing_transformer = CategoricalMissingValueTransformer()

# Learn statistics from training data
categorical_missing_transformer.fit(X_train)

# Transform training data
X_train_cat = categorical_missing_transformer.transform(X_train)

In [12]:
categorical_features_to_check = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType",
    "Electrical"
]

X_train_cat[categorical_features_to_check].isnull().sum()

PoolQC          0
MiscFeature     0
Alley           0
Fence           0
FireplaceQu     0
GarageType      0
GarageFinish    0
GarageQual      0
GarageCond      0
BsmtQual        0
BsmtCond        0
BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
MasVnrType      0
Electrical      0
dtype: int64

---

## Step 13: Building the Feature Engineering Transformer

The next component of the preprocessing pipeline is the **Feature Engineering Transformer**.

During exploratory data analysis, several new features were created to better represent meaningful real-world characteristics of a house. These engineered features combine information from existing variables, making the dataset more informative for machine learning models.

The engineered features for this project are:

- **HouseAge** – Represents the age of the house at the time of sale.
- **TotalIndoorArea** – Represents the total usable indoor living area.
- **TotalBathrooms** – Represents the overall bathroom availability using weighted contributions from full and half bathrooms.

Since the engineered feature **HouseAge** already captures the information contained in `YrSold`, the original feature `YrSold` is removed to avoid redundancy.

Unlike imputers or scalers, feature engineering does not learn any parameters from the data. Therefore, the `fit()` method simply returns the transformer itself, while all feature creation logic is implemented inside the `transform()` method.

Placing feature engineering after missing value handling ensures that all engineered features are computed from clean and consistent data.

---

### Creating Engineered Features

The `transform()` method is responsible for creating the engineered features identified during exploratory data analysis.

The following features are created:

- **HouseAge** – Calculates the age of the house at the time of sale.
- **TotalIndoorArea** – Combines the above-ground living area and basement area to represent the total usable indoor space.
- **TotalBathrooms** – Combines full and half bathrooms using weighted contributions, where each half bathroom contributes 0.5.

After creating these features, the original feature `YrSold` is removed because its information is fully represented by the engineered feature `HouseAge`.

Since feature engineering does not require learning any parameters from the data, the `fit()` method simply returns the current transformer object using `return self`.

Returning `self` follows the standard Scikit-learn estimator API, where every estimator returns the fitted estimator after calling `fit()`. This ensures a consistent interface across all Scikit-learn transformers and allows the custom transformer to integrate seamlessly with components such as `Pipeline` and `GridSearchCV`, while also enabling method chaining (e.g., `fit().transform()`).

The `transform()` method then applies the same feature engineering logic consistently to the training, validation, test, and future production datasets.

---

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):

        X = X.copy()

      
        # HouseAge

        X["HouseAge"] = X["YrSold"] - X["YearBuilt"]

        # TotalIndoorArea
        X["TotalIndoorArea"] = (
            X["GrLivArea"] +
            X["TotalBsmtSF"]
        )

        # TotalBathrooms

        X["TotalBathrooms"] = (
            X["FullBath"]
            + 0.5 * X["HalfBath"]
            + X["BsmtFullBath"]
            + 0.5 * X["BsmtHalfBath"]
        )

        # Drop Redundant Feature

        X = X.drop(columns="YrSold")

        return X

In [14]:
# Create transformer
feature_engineering_transformer = FeatureEngineeringTransformer()

# Fit (no learning happens, but follows sklearn API)
feature_engineering_transformer.fit(X_train)

# Transform
X_train_fe = feature_engineering_transformer.transform(X_train)

# Verify new features
X_train_fe[
    [
        "HouseAge",
        "TotalIndoorArea",
        "TotalBathrooms"
    ]
].head()

,HouseAge,TotalIndoorArea,TotalBathrooms
254,53,2628,2.0
1066,16,2370,2.5
638,98,1592,1.0
799,70,2499,2.5
380,86,2717,2.0


---

## Step 14: Building the Dataset-Level Preprocessing Pipeline

Before separating numerical and categorical features, certain preprocessing operations must be applied to the **entire dataset**.

Some preprocessing tasks require access to both numerical and categorical features simultaneously. For example, imputing missing values in `LotFrontage` requires the categorical feature `Neighborhood` to compute the neighborhood-wise median.

Therefore, these preprocessing operations cannot be placed inside a numerical-only preprocessing pipeline.

To preserve these feature dependencies, the following transformers are organized into a single dataset-level preprocessing pipeline:

1. **Numerical Missing Value Transformer**
2. **Categorical Missing Value Transformer**
3. **Feature Engineering Transformer**

The pipeline executes these transformers sequentially, producing a clean and consistent dataset.

Once dataset-level preprocessing is complete, the transformed dataset can safely be separated into numerical and categorical features for type-specific preprocessing such as feature scaling and one-hot encoding.

This two-stage architecture ensures that:

- Preprocessing steps have access to all required features.
- Dependencies between numerical and categorical features are preserved.
- The preprocessing workflow remains modular, reusable, and fully compatible with Scikit-learn pipelines.


---

In [15]:
dataset_preprocessing_pipeline = Pipeline(
    steps=[
        (
            "numerical_missing_values",
            NumericalMissingValueTransformer()
        ),
        (
            "categorical_missing_values",
            CategoricalMissingValueTransformer()
        ),
        (
            "feature_engineering",
            FeatureEngineeringTransformer()
        )
    ]
)

### Note

This dataset-level preprocessing pipeline is intended to be composed with the final preprocessing workflow. Although it can be used independently, its primary role is to perform dataset-level preprocessing before the `ColumnTransformer` applies type-specific transformations.

The `FeatureEngineeringTransformer` does not learn any parameters during `fit()`. However, Scikit-learn verifies whether an estimator has been fitted using `check_is_fitted()`, which typically expects at least one fitted attribute (an attribute ending with `_`).

To ensure full compatibility with the Scikit-learn API and to prevent potential `NotFittedError` exceptions when the transformer is used inside pipelines, a dummy fitted attribute (e.g., `is_fitted_` or `n_features_in_`) is created during the `fit()` method.

---

## Step 15: Building the Column Transformer

After completing the dataset-level preprocessing, the dataset contains clean features along with the newly engineered features. The next step is to apply preprocessing operations that are **specific to each feature type**.

### Why is a Column Transformer Required?

Numerical and categorical features require different preprocessing techniques before they can be used by a machine learning model.

- **Numerical features** need to be standardized so that they are on a comparable scale.
- **Categorical features** need to be converted into numerical representations using one-hot encoding.

Applying the same preprocessing operation to every feature is neither appropriate nor efficient. Therefore, a mechanism is required to apply different transformations to different subsets of features.

### Using `ColumnTransformer`

Scikit-learn's **`ColumnTransformer`** provides a unified way to apply different preprocessing pipelines to different feature types.

It automatically:

- **Selects** the appropriate columns.
- **Applies** the corresponding transformation.
- **Combines** the transformed outputs into a single feature matrix.

### Automatic Column Selection

Instead of manually maintaining separate lists of numerical and categorical features, this project uses **`make_column_selector()`**.

This approach automatically identifies columns based on their data type, which provides several advantages:

- **Automatically includes** newly engineered features.
- **Automatically excludes** removed features.
- **Eliminates** the need to manually update feature lists.
- **Improves maintainability** by reducing the possibility of human error.

### Conclusion

Using **`ColumnTransformer`** keeps the preprocessing workflow:

- **Modular**
- **Scalable**
- **Easy to maintain**
- **Fully compatible** with Scikit-learn pipelines

while ensuring that every feature receives the appropriate preprocessing before being passed to the machine learning model.

In [16]:
from sklearn.compose import make_column_selector
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            make_column_selector(dtype_include="number")
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            make_column_selector(dtype_include="object")
        )
    ]
)

---

## Step 16: Building the Complete Machine Learning Pipeline

The final step is to combine all preprocessing stages with the machine learning model into a single Scikit-learn pipeline.

The complete workflow consists of three sequential stages:

1. **Dataset-Level Preprocessing**
   - Handles missing values.
   - Performs feature engineering.

2. **Column Transformer**
   - Standardizes numerical features.
   - One-hot encodes categorical features.

3. **Machine Learning Model**
   - Trains a Linear Regression model using the fully preprocessed dataset.

By combining every stage into a single pipeline, the entire workflow becomes a single reusable object.

This approach provides several advantages:

- **Prevents data leakage** by ensuring that every preprocessing step is learned only from the training data.
- **Guarantees consistency** by applying identical preprocessing during training, validation, testing, and production inference.
- **Improves maintainability** since the entire workflow is managed in one place.
- **Simplifies deployment**, as the same pipeline can be saved and directly used for future predictions.

The resulting pipeline represents a complete end-to-end machine learning workflow, transforming raw input data into predictions through a single interface.

---

In [17]:
#Step 16.1: Build the Final Pipeline
model_pipeline = Pipeline(
    steps=[
        (
            "dataset_preprocessing",
            dataset_preprocessing_pipeline
        ),
        (
            "column_transformer",
            preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)

In [18]:
#Step 16.2: Train the Pipeline
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dataset_preprocessing', ...), ('column_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('numerical_missing_values', ...), ('categorical_missing_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
lotfrontage_median_,"Series[float64](25,)",Neighborhood ...dtype: float64
Name,Type,Value


In [19]:
#Step 16.3: Make Predictions
y_train_pred = model_pipeline.predict(X_train)

y_valid_pred = model_pipeline.predict(X_valid)

## Prediction Workflow

After the pipeline has been trained using `pipeline.fit(X_train, y_train)`, all preprocessing parameters are learned **only from the training data** and stored inside the fitted pipeline.

When a new dataset is passed to:

```python
pipeline.predict(X_new)
```

the pipeline automatically executes the following workflow:

```text
Raw Input Data
      │
      ▼
NumericalMissingValueTransformer.transform()
      │
      ▼
CategoricalMissingValueTransformer.transform()
      │
      ▼
FeatureEngineeringTransformer.transform()
      │
      ▼
ColumnTransformer.transform()
      │
      ├── StandardScaler.transform()
      │      (uses mean and standard deviation learned from training)
      │
      └── OneHotEncoder.transform()
             (uses categories learned from training)
      │
      ▼
LinearRegression.predict()
      │
      ▼
Predicted House Price
```

### Key Points

- `predict()` **does not call `fit()`** on any transformer or the model.
- No preprocessing statistics are recalculated during prediction.
- Missing values, scaling, encoding, and feature engineering all use the parameters learned during `pipeline.fit(X_train, y_train)`.
- This guarantees that every new dataset is processed exactly the same way as the training data, ensuring consistency and preventing data leakage.

---

In [20]:
#Step 16.4: Verify Predictions
print("Training Predictions Shape :", y_train_pred.shape)
print("Validation Predictions Shape:", y_valid_pred.shape)

Training Predictions Shape : (1168,)
Validation Predictions Shape: (292,)


In [21]:
#Step 16.5: Inspect Sample Predictions
import pandas as pd

pd.DataFrame({
    "Actual": y_valid.values[:10],
    "Predicted": y_valid_pred[:10]
})

,Actual,Predicted
0,154500,160238.128656
1,325000,342808.285804
2,115000,90242.455731
3,159000,176714.347623
4,315500,321295.321127
5,75500,68063.497948
6,311500,239692.356451
7,146000,146451.210239
8,84500,62814.657773
9,135500,151698.645032


---

#  Summary

In this notebook, the manual preprocessing workflow developed during the previous phase of the project was converted into an automated machine learning pipeline using Scikit-learn.

The preprocessing steps were encapsulated into custom transformers following Scikit-learn's `fit()` and `transform()` interface. This allows the same preprocessing logic to be consistently applied during both model training and inference.

The following preprocessing tasks were implemented:

- Numerical missing value imputation
- Categorical missing value imputation
- Correction of dataset inconsistencies identified during EDA
- Feature engineering
- Feature removal after engineering

For preprocessing that required learning information from the data (such as neighborhood-wise medians and categorical modes), the required parameters were learned only from the training dataset during the `fit()` stage and reused during `transform()`, preventing data leakage.

After dataset-level preprocessing, a `ColumnTransformer` was implemented to perform feature-type-specific preprocessing by:

- Standardizing numerical features using `StandardScaler`
- Encoding categorical features using `OneHotEncoder`

Instead of maintaining separate lists of numerical and categorical features, `make_column_selector()` was used to automatically identify feature types based on their data type.

Finally, the preprocessing workflow was combined with a **baseline Linear Regression model** using Scikit-learn's `Pipeline`. As a result, preprocessing and model training are executed through a single workflow, ensuring that the same preprocessing steps are applied whenever the model is trained or used for prediction.

---

## Components Implemented

- Custom numerical missing value transformer
- Custom categorical missing value transformer
- Custom feature engineering transformer
- Dataset-level preprocessing pipeline
- Column-wise preprocessing using `ColumnTransformer`
- Baseline Linear Regression model
- End-to-end Scikit-learn machine learning pipeline

---



## Pipeline Workflow

```text
Raw Dataset
      │
      ▼
Dataset-Level Preprocessing Pipeline
│
├── Numerical Missing Value Handling
├── Categorical Missing Value Handling
└── Feature Engineering
      │
      ▼
ColumnTransformer
│
├── Numerical Pipeline
│     └── StandardScaler
│
└── Categorical Pipeline
      └── OneHotEncoder
      │
      ▼
Baseline Linear Regression Model
      │
      ▼
Predictions
```
```

This notebook establishes the complete preprocessing and training workflow. The next stage of the project focuses on evaluating the baseline model, analyzing its performance, and comparing it with other regression algorithms.

----

## Testing the Reusable Machine Learning Pipeline
I use the src folder to store reusable source code. My notebooks are used for experimentation, analysis, and documenting results, while src contains reusable implementation such as custom transformers and pipeline creation. This separation improves maintainability, avoids code duplication, and makes it easy to reuse the same implementation across multiple notebooks or scripts. It also follows common Python and machine learning project conventions

### Project Setup

Before importing reusable modules, add the project root directory to Python's search path.

```python
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))
```

This allows Python to locate the `src` package when running notebooks from the `notebooks/` directory.

---

In [31]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))


---



### Importing the Reusable Pipeline

Import the reusable pipeline using an **absolute import**:

```python
from src.pipeline import create_pipeline
```


### Why Not Use Relative Imports (`..`)?

Relative imports such as:

```python
from ..src.pipeline import create_pipeline
```

do not work in Jupyter notebooks because notebooks are executed as **standalone files**, not as part of a Python package. Since a notebook has no parent package, Python cannot resolve the `..` reference.

Therefore, **absolute imports** are the recommended and standard approach for notebook-based machine learning projects.

---

In [32]:
from src.pipeline import create_pipeline



In [34]:
#Create the Pipeline
model_pipeline = create_pipeline(LinearRegression())

In [36]:
#Train
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dataset_preprocessing', ...), ('column_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('numerical_missing_values', ...), ('categorical_missing_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
lotfrontage_median_,"Series[float64](25,)",Neighborhood ...dtype: float64
Name,Type,Value


In [37]:
y_train_pred = model_pipeline.predict(X_train)

y_valid_pred = model_pipeline.predict(X_valid)

In [38]:
#Step 16.5: Inspect Sample Predictions
import pandas as pd

pd.DataFrame({
    "Actual": y_valid.values[:10],
    "Predicted": y_valid_pred[:10]
})

,Actual,Predicted
0,154500,160238.128656
1,325000,342808.285804
2,115000,90242.455731
3,159000,176714.347623
4,315500,321295.321127
5,75500,68063.497948
6,311500,239692.356451
7,146000,146451.210239
8,84500,62814.657773
9,135500,151698.645032
